# Time offsets

This notebook introduces the concept of _time offsets_ and how they are used in `gwrefpy`.

This notebook can be downloaded from the source code [here](https://github.com/andersretznerSGU/gwrefpy/blob/main/docs/user_guide/3_time_offsets.ipynb).

Time offsets are a concept to handle time series with different timestamps. If two time series share the same timestamps, they can be compared directly. If not, we need to introduce a time offset to align them. The offset represents an interval for which all data points will be considered as having the same timestamp.

In the following notebook, we will illustrate this concept with a simple example. The example is inspired by section _3.3_ in [Strandanger (2024)](https://svenskageotekniskaforeningen.se/wp-content/uploads/Publikationer/SGF_Rapporter/2024_2_Akvifars_refmetod.pdf).


In [ ]:
import gwrefpy as gr
import pandas as pd
import matplotlib.pyplot as plt
import warnings

gr.set_log_level("CRITICAL")
warnings.filterwarnings("ignore")

# Data

Let's start by declaring the data we will explore as an example. We create `Well` objects and add to a `Model` object. Finally, we plot the data.

In [ ]:
obs_ts = pd.Series(
    index=[
        pd.Timestamp("2023-01-07"),
        pd.Timestamp("2023-02-01"),
        pd.Timestamp("2023-02-25"),
    ],
    data=[10.4, 10.7, 10.8],
    name="obs",
)
ref_ts = pd.Series(
    index=[
        pd.Timestamp("2023-01-08"),
        pd.Timestamp("2023-02-03"),
        pd.Timestamp("2023-02-09"),
        pd.Timestamp("2023-02-25"),
        pd.Timestamp("2023-02-28"),
    ],
    data=[8.9, 9.2, 9.3, 9.3, 9.5],
    name="ref",
)
obs = gr.Well("obs", False, obs_ts)
ref = gr.Well("ref", True, ref_ts)

model = gr.Model("offset test")
model.add_well([obs, ref])

In [ ]:
fig, (up, down) = plt.subplots(nrows=2, sharex=True)

obs_ts.plot(ax=up, marker="o", linestyle="", label="obs", color="black")
ref_ts.plot(ax=down, marker="o", linestyle="", label="ref", color="blue")
_ = [ax.legend() for ax in (up, down)]
_ = [ax.grid() for ax in (up, down)]

As we can see, the wells share some timestamps, but not all. In fact, only one timestamp (2023-02-25) is shared between the two wells. The table below shows an outer join between the two wells.

In [ ]:
pd.concat([obs_ts, ref_ts], axis="columns", join="outer")

# Using time offsets

Our two wells have only one shared timestamp, but several timestamps that are close to each other. To compare the two time series, we can introduce a time offset.

A time offset defines a `±offset` window around each observation time point. All reference points falling within that window are aggregated (mean by default) into a single matched value for that observation. This is called the **anchor method**, and it is the default since version 1.0.0.

```{important}
The offset extends in both directions. For example, given an observation timestamp of 2023-02-25 and an offset of 3 days, all reference points between 2023-02-22 and 2023-02-28 will be matched to that observation.
```

The offset argument can be given as a string or a `pandas.Timedelta` object. Refer to the [pandas documentation](https://pandas.pydata.org/docs/reference/api/pandas.Timedelta.html) for more information about accepted strings.

As of version 0.2.0 the aggregation within an offset can be set to `"mean"` (default), `"median"`, `"min"`, or `"max"` (see the API reference for the `Model` class and its `fit` method).

## Grouping methods

Two grouping methods are available via the `method` parameter:

- **`"anchor"` (default)**: each observation time is the center of a `±offset` window; all reference points inside that window are aggregated into one output pair per observation. Works reliably for both sparse and dense/high-frequency reference data.
- **`"consecutive"`**: all timestamps from both series are merged, sorted, and split into groups wherever the gap between consecutive timestamps exceeds `offset`. Both reference and observation values within each group are aggregated. This was the default prior to version 1.0.0, and remains available for very sparse, irregular monitoring data where the chaining behaviour is intentional.

## Analysing offsets

We provide a helper function for analysing multiple time offsets at once: `analyze_offsets`. This function takes a list of offsets and returns the number of matching data pairs. `analyze_offsets(obs, ref, ...)` uses the **obs** timestamps as anchors, so it answers the question: *"for how many observation measurements can a matching reference be found?"* Both methods are supported via the `method` parameter (default `"anchor"`).

Let's analyse offsets of 0, 1, and 3 days.

In [ ]:
offsets = ["0D", "1D", "3D"]
results = gr.analyze_offsets(obs, ref, offsets)

In [ ]:
results = results.reset_index().set_index("index", drop=True)
results.index.name = "offset"

In [ ]:
results

Let's plot the width of the offset to understand the results better.

In [ ]:
fig, axs = plt.subplots(nrows=len(offsets), ncols=1, sharex=True)

for i, (ax, offset) in enumerate(zip(axs, offsets)):
    ax2 = ax.twinx()
    
    for ts, value in obs_ts.items():
        x_start = ts - pd.Timedelta(offset)
        x_end = ts + pd.Timedelta(offset)
        ax.plot([x_start, x_end], [value, value], 
                color="gray", alpha=0.75, linewidth=5)
    
    for ts, value in ref_ts.items():
        x_start = ts - pd.Timedelta(offset)
        x_end = ts + pd.Timedelta(offset)
        ax2.plot([x_start, x_end], [value, value], 
                 color="blue", alpha=0.5, linewidth=5)
    
    obs_ts.plot(ax=ax, marker="o", linestyle="", label="obs", color="black")
    ref_ts.plot(ax=ax2, marker="o", linestyle="", label="ref", color="blue")
    
    ax.set_ylabel("obs", color="black")
    ax2.set_ylabel("ref", color="blue")
    
    ax.tick_params(axis="y", labelcolor="black")
    ax2.tick_params(axis="y", labelcolor="blue")

    ax.set_title(f"{pd.to_timedelta(offset).days} days")
    ax.grid()

fig.tight_layout()

For this simple example, a time offset of 3 days seems reasonable.

With the default **anchor method** and `offset="3D"`, each observation timestamp is the center of a `±3D` window:

- **Period 1** (January): obs `2023-01-07` window spans Jan 4–10. Ref `2023-01-08` is 1 day away → one matched pair.
- **Period 2** (early February): obs `2023-02-01` window spans Jan 29–Feb 4. Ref `2023-02-03` is 2 days away → matched. Ref `2023-02-09` is 8 days away → not matched. One matched pair.
- **Period 3** (late February): obs `2023-02-25` window spans Feb 22–28. Both ref `2023-02-25` (0 days) and ref `2023-02-28` (3 days) fall inside → **both are aggregated into one pair** (mean of 9.3 and 9.5 = 9.4).

Fitting produces **3 data pairs** — one per observation point.


In [ ]:
offset = "3D"

fig, ax = plt.subplots(figsize=(6, 3))
ax2 = ax.twinx()

for ts, value in obs_ts.items():
    x_start = ts - pd.Timedelta(offset)
    x_end = ts + pd.Timedelta(offset)
    ax.plot([x_start, x_end], [value, value], 
            color="gray", alpha=0.5, linewidth=5)

for ts, value in ref_ts.items():
    x_start = ts - pd.Timedelta(offset)
    x_end = ts + pd.Timedelta(offset)
    ax2.plot([x_start, x_end], [value, value], 
             color="blue", alpha=0.5, linewidth=5)

obs_ts.plot(ax=ax, marker="o", linestyle="", label="obs", color="black")
ref_ts.plot(ax=ax2, marker="o", linestyle="", label="ref", color="blue")

ax.set_ylabel("obs", color="black")
ax2.set_ylabel("ref", color="blue")

ax.tick_params(axis="y", labelcolor="black")
ax2.tick_params(axis="y", labelcolor="blue")

periods = [
    ("2023-01-05", "2023-01-20", "Period 1"),
    ("2023-01-28", "2023-02-12", "Period 2"),
    ("2023-02-20", "2023-03-05", "Period 3")
]

colors = ["lightblue", "lightgreen", "grey"]
y_positions = [0.9, 0.1, 0.1]

for i, (start_date, end_date, label) in enumerate(periods):
    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)
    
    ax.axvspan(start, end, alpha=0.25, color=colors[i], zorder=0)
    
    center = start + (end - start) / 2
    y_min, y_max = ax.get_ylim()
    y_pos = y_min + (y_max - y_min) * y_positions[i]
    ax.text(center, y_pos, label, 
            ha="center", va="center", fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

ax.grid()

plt.xticks(rotation=45)
plt.tight_layout()

When fitting these two wells with an offset of 3 days, three data pairs will formed as explained above.

# Shifting timeseries
It is possible to shift the timeseries of an observation well to account for delays in the response.

We will import the deviations example and shift the observation well with 10 days.

In [ ]:
model = gr.Model("small_example.gwref")
model.fits.clear()

obs_well = model.wells.obs()[0]
ref_well = model.wells.ref()[0]
# shift the time for this example
shift = pd.Timedelta("-10D")
obs_well.timeseries = obs_well.shift_timeseries(shift)

_ = gr.plots.compare(model.wells)

The fit of the two wells are now

In [ ]:
fit0 = model.fit(obs_well, ref_well, offset="0D")
_ = fit0.plots.diagnostics()
_ = fit0.plot()

and we can compare this fit with another one with a shift

In [ ]:
fit1 = model.fit(obs_well, ref_well, offset="0D", shift="10D")
_ = fit1.plots.diagnostics()
_ = fit1.plot()

Plotting both fits together shows the effect of the shift.

In [ ]:
_ = gr.plots.compare([fit0, fit1])

This concludes this notebook on time offsets in `gwrefpy`. Happy fitting!